# 05 - Database

Guardar los DataFrames enriquecidos en SQLite para consultas SQL y el dashboard.

**Secciones:**
- 5.1 Cargar datos y aplicar enrichment
- 5.2 Crear base de datos SQLite
- 5.3 Guardar inventory en SQLite
- 5.4 Guardar orders en SQLite
- 5.5 Consultas SQL basicas
- 5.6 Consultas SQL de negocio
- 5.7 Crear tabla summary
- 5.8 Verificar integridad
- 5.9 Conclusion

---
## 5.1 - Cargar datos y aplicar enrichment

Mismo proceso de 02_clean + 03_enrich para que el notebook sea independiente.

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# --- Cargar datos ---
PATH_FILE = "../data/inventory_anon.xlsx"
sheets = pd.read_excel(PATH_FILE, sheet_name=["Inventory by Storeroom", "OPEN ORDER"])
inv = sheets["Inventory by Storeroom"]
orders = sheets["OPEN ORDER"]

# --- Limpieza (02_clean) ---
null_per_row = inv.isnull().sum(axis=1)
inv = inv.drop(null_per_row[null_per_row > 50].index)

null_pct = inv.isnull().sum() / len(inv)
junk_cols = null_pct[null_pct == 1.0].index.tolist()
junk_cols.append("Unnamed: 61")
inv = inv.drop(columns=junk_cols)

null_pct_orders = orders.isnull().sum() / len(orders)
junk_cols_orders = null_pct_orders[null_pct_orders == 1.0].index.tolist()
orders = orders.drop(columns=junk_cols_orders)

# --- Enrichment (03_enrich) ---
inv["Dead Stock"] = ((inv["Balance On Hand"] > 0) & (inv["Average Daily Usage"] == 0)).astype(int)

has_usage = inv["Avg12Mon Usage"] > 0
excess = inv["Balance On Hand"] > (inv["Avg12Mon Usage"] * 12)
inv["Overstock"] = (has_usage & excess).astype(int)

sorted_inv = inv.sort_values("Inventory Value", ascending=False)
sorted_inv["Cumulative Value"] = sorted_inv["Inventory Value"].cumsum()
total_value = sorted_inv["Inventory Value"].sum()
sorted_inv["Cumulative Pct"] = sorted_inv["Cumulative Value"] / total_value
sorted_inv["ABC"] = "C"
sorted_inv.loc[sorted_inv["Cumulative Pct"] <= 0.95, "ABC"] = "B"
sorted_inv.loc[sorted_inv["Cumulative Pct"] <= 0.80, "ABC"] = "A"
inv["ABC"] = sorted_inv["ABC"]

inv["DOS"] = np.where(
    inv["Average Daily Usage"] > 0,
    inv["Balance On Hand"] / inv["Average Daily Usage"],
    np.inf
)

active = inv["Average Daily Usage"] > 0
low_stock = inv["Balance On Hand"] < (inv["Average Daily Usage"] * 30)
inv["Reorder Risk"] = (active & low_stock).astype(int)

orders["Open Value"] = orders["Open Qty"] * orders["Price"]
orders["Redundant Order"] = (orders["BOH"] >= orders["Open Qty"]).astype(int)
orders["Issue Date"] = pd.to_datetime(orders["Issue Date"], errors="coerce")
last_date = orders["Issue Date"].max()
orders["Age Days"] = (last_date - orders["Issue Date"]).dt.days

print(f"Inventory: {inv.shape}")
print(f"Orders: {orders.shape}")

Inventory: (16249, 64)
Orders: (3813, 18)


---
## 5.2 - Crear base de datos SQLite

**SQLite** es una base de datos que se guarda en un solo archivo (`.db`). No necesita servidor — ideal para proyectos locales y prototipos.

**SQLAlchemy** es una libreria de Python que conecta pandas con bases de datos. Permite guardar DataFrames como tablas SQL con una sola linea de codigo.


DataFrame (pandas) ---> SQLAlchemy ---> SQLite (archivo .db)


In [2]:
# create_engine crea la conexion a la base de datos
# "sqlite:///" + ruta = usa SQLite y guarda en ese archivo
# si el archivo no existe, SQLite lo crea automaticamente
DB_PATH = "../data/inventory.db"
engine = create_engine(f"sqlite:///{DB_PATH}")

print(f"Base de datos: {DB_PATH}")
print("Conexion creada")

Base de datos: ../data/inventory.db
Conexion creada


---
## 5.3 - Guardar inventory en SQLite

`to_sql()` convierte un DataFrame en una tabla SQL. 

In [3]:
# limpia nombres de columnas para SQL: espacios -> guion bajo, todo minusculas
inv.columns = inv.columns.str.strip().str.lower().str.replace(" ", "_")
inv.head(3)

,region_name,country_name,operation_name,plant_code,plant_name,currency,exchange_rate,blanket_number,commodity_code,commodity,...,total_on_order,last_modified,in_erp,average_daily_usage,trend,dead_stock,overstock,abc,dos,reorder_risk
0,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,MACHINE SPARES,2.0,...,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0,0,C,inf,0
1,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0,0,C,inf,0
2,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0,0,C,inf,0


In [4]:
# to_sql() guarda el DataFrame como tabla en la base de datos
# "inventory" es el nombre de la tabla
# engine = la conexion que creada arriba
# if_exists="replace" = si la tabla ya existe, la sobreescribe (util para re-ejecutar)
# index=False = no guarda el indice de pandas como columna extra
inv.to_sql("inventory", engine, if_exists="replace", index=False)

print(f"Tabla 'inventory' creada: {len(inv):,} filas, {len(inv.columns)} columnas")

Tabla 'inventory' creada: 16,249 filas, 64 columnas


---
## 5.4 - Guardar orders en SQLite

Mismo proceso para la tabla de ordenes abiertas.

In [5]:
# misma limpieza de nombres de columnas
orders.columns = orders.columns.str.strip().str.lower().str.replace(" ", "_")

# guarda como tabla "orders" en la misma base de datos
orders.to_sql("orders", engine, if_exists="replace", index=False)

print(f"Tabla 'orders' creada: {len(orders):,} filas, {len(orders.columns)} columnas")

Tabla 'orders' creada: 3,813 filas, 18 columnas


---
## 5.5 - queries SQL basicas

Ahora que los datos estan en SQLite, se pueden hacer consultas SQL directamente desde Python.

`pd.read_sql()` ejecuta una consulta SQL y devuelve el resultado como DataFrame.

In [6]:
# ver las tablas que existen en la base de datos
# sqlite_master es una tabla interna de SQLite que lista todas las tablas
tables = pd.read_sql("SELECT name " \
"                     FROM " \
"                         sqlite_master " \
"                     WHERE type='table'", engine)
print("Tablas en la base de datos:")
print(tables)

Tablas en la base de datos:
        name
0  inventory
1     orders


In [7]:

pd.read_sql("SELECT * FROM inventory LIMIT 5", engine)

,region_name,country_name,operation_name,plant_code,plant_name,currency,exchange_rate,blanket_number,commodity_code,commodity,...,total_on_order,last_modified,in_erp,average_daily_usage,trend,dead_stock,overstock,abc,dos,reorder_risk
0,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,MACHINE SPARES,2.0,...,0.0,2019-02-19 01:09:17.087000,N,0.0,No Trend,0,0,C,inf,0
1,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,0.0,2019-02-19 01:09:17.087000,N,0.0,No Trend,0,0,C,inf,0
2,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,0.0,2019-02-19 01:09:17.087000,N,0.0,No Trend,0,0,C,inf,0
3,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,0.0,2019-02-19 01:09:17.087000,N,0.0,No Trend,0,0,C,inf,0
4,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,0.0,2019-02-19 01:09:17.087000,N,0.0,No Trend,0,0,C,inf,0


In [8]:
# COUNT(*) = cuenta el total de filas en la tabla
# es como len(df) en pandas
pd.read_sql("SELECT COUNT(*) AS total_filas FROM inventory", engine)

,total_filas
0,16249


---
## 5.6 - Queries SQL de negocio

Consultas que responden preguntas reales del inventario - las mismas que se hicieron con pandas en notebooks anteriores, pero ahora en SQL.

In [9]:
# valor total de dead stock por categoria ABC
# SUM() = suma valores, WHERE filtra filas, GROUP BY agrupa
# ORDER BY ordena el resultado (DESC = de mayor a menor)
query_dead = """
SELECT 
    abc,
    COUNT(*) AS items,
    ROUND(SUM(inventory_value), 0) AS dead_stock_value
FROM inventory
WHERE dead_stock = 1
GROUP BY abc
ORDER BY dead_stock_value DESC
"""
pd.read_sql(query_dead, engine)

,abc,items,dead_stock_value
0,A,600,4775385.0
1,B,723,662554.0
2,C,2472,213958.0


In [10]:
# top 10 proveedores con mas valor en ordenes abiertas
# esta es la misma consulta que el aging de 04_analytics, pero en SQL
query_aging = """
SELECT 
    supplier_name,
    COUNT(*) AS orders_count,
    ROUND(AVG(age_days), 0) AS avg_age_days,
    MAX(age_days) AS max_age_days,
    ROUND(SUM(open_value), 0) AS total_open_value,
    SUM(redundant_order) AS redundant_count
FROM orders
GROUP BY supplier_name
ORDER BY total_open_value DESC
LIMIT 10
"""
pd.read_sql(query_aging, engine)

,supplier_name,orders_count,avg_age_days,max_age_days,total_open_value,redundant_count
0,SUPPLIER-001,835,139.0,711,1110774.0,476
1,SUPPLIER-026,33,145.0,403,831704.0,7
2,SUPPLIER-004,42,304.0,410,708639.0,28
3,SUPPLIER-013,428,56.0,641,442951.0,314
4,SUPPLIER-010,253,97.0,707,331200.0,144
5,SUPPLIER-005,162,209.0,979,252992.0,70
6,SUPPLIER-059,242,37.0,259,251878.0,235
7,SUPPLIER-120,3,30.0,83,190864.0,1
8,SUPPLIER-021,39,86.0,466,183528.0,31
9,SUPPLIER-029,3,235.0,367,179216.0,0


In [11]:
# items A en riesgo de desabasto — los mas urgentes de reordenar
# WHERE con AND combina multiples condiciones (como & en pandas)
query_risk = """
SELECT 
    store_room,
    commodity_code,
    balance_on_hand AS boh,
    average_daily_usage AS daily_usage,
    ROUND(dos, 0) AS days_of_supply,
    ROUND(inventory_value, 0) AS value
FROM inventory
WHERE reorder_risk = 1 AND abc = 'A'
ORDER BY inventory_value DESC
LIMIT 10
"""
pd.read_sql(query_risk, engine)

,store_room,commodity_code,boh,daily_usage,days_of_supply,value
0,ROOM-4,PERISHABLE,1690.0,69.130435,24.0,16021.0
1,ROOM-4,PERISHABLE,157.0,6.532609,24.0,9302.0
2,ROOM-4,PERISHABLE,32.0,1.238356,26.0,9277.0
3,ROOM-4,PERISHABLE,900.0,40.326087,22.0,8613.0
4,ROOM-3,GENERAL SUPPLIES,243.0,12.000000,20.0,7044.0
5,ROOM-2,PERISHABLE,560.0,26.054795,21.0,6742.0
6,ROOM-4,EXPENSE,820.0,40.380435,20.0,6527.0
7,ROOM-4,PERISHABLE,1665.0,119.402174,14.0,6044.0
8,ROOM-4,PERISHABLE,264.0,16.630435,16.0,5853.0
9,ROOM-4,PERISHABLE,510.0,24.673913,21.0,5687.0


---
## 5.7 - Crear tabla summary

Una tabla resumen con KPIs pre-calculados. El dashboard leera esta tabla para mostrar metricas sin recalcular cada vez.

In [12]:
# crea un diccionario con los KPIs principales del inventario
# cada key = nombre del KPI, cada value = el valor calculado
summary = {
    "total_items": [len(inv)],
    "total_value": [round(inv["inventory_value"].sum(), 0)],
    "dead_stock_items": [int(inv["dead_stock"].sum())],
    "dead_stock_value": [round(inv.loc[inv["dead_stock"] == 1, "inventory_value"].sum(), 0)],
    "overstock_items": [int(inv["overstock"].sum())],
    "overstock_value": [round(inv.loc[inv["overstock"] == 1, "inventory_value"].sum(), 0)],
    "reorder_risk_items": [int(inv["reorder_risk"].sum())],
    "reorder_risk_value": [round(inv.loc[inv["reorder_risk"] == 1, "inventory_value"].sum(), 0)],
    "total_orders": [len(orders)],
    "redundant_orders": [int(orders["redundant_order"].sum())],
    "redundant_value": [round(orders.loc[orders["redundant_order"] == 1, "open_value"].sum(), 0)],
    "orders_over_1yr": [int((orders["age_days"] > 365).sum())]
}

# convierte el diccionario a DataFrame (1 fila con todos los KPIs)
df_summary = pd.DataFrame(summary)

# guarda como tabla "summary" en la base de datos
df_summary.to_sql("summary", engine, if_exists="replace", index=False)

print("Tabla 'summary' creada con KPIs:")
# .T transpone el DataFrame — convierte columnas en filas para mejor lectura
df_summary.T

Tabla 'summary' creada con KPIs:


,0
total_items,16249.0
total_value,9150387.0
dead_stock_items,3795.0
dead_stock_value,5651897.0
overstock_items,1234.0
overstock_value,1915385.0
reorder_risk_items,1871.0
reorder_risk_value,390324.0
total_orders,3813.0
redundant_orders,2538.0


---
## 5.8 - Verificar integridad

Confirmar que los datos en SQLite coinciden con los DataFrames originales.

In [13]:
# lee las tablas de vuelta desde SQLite para comparar
db_inv = pd.read_sql("SELECT COUNT(*) AS n FROM inventory", engine)
db_ord = pd.read_sql("SELECT COUNT(*) AS n FROM orders", engine)
db_sum = pd.read_sql("SELECT COUNT(*) AS n FROM summary", engine)

# verifica que las cantidades coincidan con los DataFrames originales
print("=== Verificacion de integridad ===")
print(f"Inventory: pandas={len(inv):,} | SQLite={db_inv['n'][0]:,} | {'OK' if len(inv) == db_inv['n'][0] else 'ERROR'}")
print(f"Orders:    pandas={len(orders):,} | SQLite={db_ord['n'][0]:,} | {'OK' if len(orders) == db_ord['n'][0] else 'ERROR'}")
print(f"Summary:   pandas=1 | SQLite={db_sum['n'][0]} | {'OK' if db_sum['n'][0] == 1 else 'ERROR'}")

=== Verificacion de integridad ===
Inventory: pandas=16,249 | SQLite=16,249 | OK
Orders:    pandas=3,813 | SQLite=3,813 | OK
Summary:   pandas=1 | SQLite=1 | OK


In [14]:
# ver el tamano del archivo .db en disco
import os

# os.path.getsize() retorna el tamano en bytes
# / 1024 / 1024 convierte bytes → kilobytes → megabytes
db_size = os.path.getsize(DB_PATH) / 1024 / 1024
print(f"Tamano de la base de datos: {db_size:.1f} MB")

Tamano de la base de datos: 6.0 MB


---
## 5.9 - Conclusion

### Lo que se hizo

Se migraron los 2 DataFrames enriquecidos (inventory + orders) a una base de datos SQLite con 3 tablas:

| Tabla | Filas | Descripcion |
|-------|-------|-------------|
| `inventory` | 16,249 | Inventario limpio + flags (Dead Stock, Overstock, ABC, DOS, Reorder Risk) |
| `orders` | 3,813 | Ordenes abiertas + flags (Open Value, Redundant Order, Age Days) |
| `summary` | 1 | KPIs pre-calculados para el dashboard |

### Por que SQLite

- **Sin servidor** — es un archivo .db, no necesita instalar nada
- **Portable** — se copia y se mueve como cualquier archivo
- **Compatible** — se puede cambiar a SQL Server o PostgreSQL cambiando solo la linea del create_engine()
- **Rapido** — para 20K filas es instantaneo

